# Lecture 11 — Capstone: Topology and Majorana Edge Modes

---

## Overview

The Kitaev chain is the final lecture in this series. It brings together everything that came before — phase transitions, energy gaps, exact diagonalisation, and tensor networks — while introducing something genuinely new: **topological order**.

The two phases of the Kitaev chain look the same from the perspective of any local order parameter. The distinction is **global** — a property of the entire ground state wavefunction that cannot be changed without closing the energy gap. At the phase boundary, **Majorana zero modes** appear: exotic quasiparticles that are their own antiparticles and carry quantum information protected from local errors.

---

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

## 1. The Kitaev Chain Hamiltonian

$$\hat{H} = -\mu \sum_i \hat{c}^\dagger_i \hat{c}_i - t \sum_i (\hat{c}^\dagger_{i+1} \hat{c}_i + \text{h.c.}) + \Delta \sum_i (\hat{c}_{i+1} \hat{c}_i + \text{h.c.})$$

Parameters: $\mu$ (chemical potential), $t$ (hopping), $\Delta$ (p-wave pairing). The pairing term destroys two fermions simultaneously — the essential physics of a 1D p-wave superconductor.

We implement this using the Jordan-Wigner transformation to map spinless fermions to Pauli operators.

---

In [ ]:
def build_kitaev(N, mu, t, Delta):
    """
    Build the Kitaev chain Hamiltonian via Jordan-Wigner transformation.
    c_i = (prod_{j<i} sigma_z_j) * sigma_minus_i
    Returns a real sparse Hamiltonian matrix.
    """
    dim = 2**N
    H = sp.lil_matrix((dim, dim), dtype=complex)

    def jw_string(state, site):
        """Jordan-Wigner string: product of (-1)^{n_j} for j < site."""
        phase = 1
        for j in range(site):
            if (state >> j) & 1:
                phase *= -1
        return phase

    for state in range(dim):
        # Chemical potential: -mu * c†_i c_i
        for i in range(N):
            if (state >> i) & 1:  # site i occupied
                H[state, state] -= mu

        for i in range(N - 1):
            occ_i = (state >> i) & 1
            occ_j = (state >> (i + 1)) & 1

            # Hopping: -t * (c†_{i+1} c_i + h.c.)
            if occ_i == 1 and occ_j == 0:
                # c†_{i+1} c_i: destroy at i, create at i+1
                new_state = state ^ (1 << i) ^ (1 << (i + 1))
                phase = jw_string(state, i) * jw_string(state ^ (1 << i), i + 1)
                H[new_state, state] -= t * phase
                H[state, new_state] -= t * phase  # h.c.

            # Pairing: Delta * (c_{i+1} c_i + h.c.)
            if occ_i == 1 and occ_j == 1:
                # c_{i+1} c_i: destroy at i and i+1
                new_state = state ^ (1 << i) ^ (1 << (i + 1))
                phase = jw_string(state, i) * jw_string(state ^ (1 << i), i + 1)
                H[new_state, state] += Delta * phase
                H[state, new_state] += Delta * phase  # h.c.

    return H.tocsr().real


# Quick test: at t=1, Delta=1, mu=0 (topological sweet spot)
N = 8
H_test = build_kitaev(N, mu=0.0, t=1.0, Delta=1.0)
evals = np.sort(np.linalg.eigvalsh(H_test.toarray()))
print(f"N={N}, topological sweet spot (μ=0, t=Δ=1):")
print(f"Lowest 4 eigenvalues: {evals[:4]}")
print(f"Ground state degeneracy gap: {evals[1] - evals[0]:.2e} (should be ~0)")

## 2. Majorana Fermions

A Majorana fermion satisfies $\hat{\gamma} = \hat{\gamma}^\dagger$. Each complex fermion decomposes into two Majoranas:

$$\hat{c}_i = \frac{1}{2}(\hat{\gamma}_{2i-1} + i\,\hat{\gamma}_{2i})$$

Rewriting the Kitaev Hamiltonian in Majorana language reveals the two phases with striking clarity: in the topological phase, $\hat{\gamma}_1$ (left end) and $\hat{\gamma}_{2N}$ (right end) are unpaired and appear nowhere in $\hat{H}$.

---

## 3. The Two Phases

**Trivial phase** ($|\mu| > 2t$): On-site Majorana pairing. Unique ground state, no boundary modes.

**Topological phase** ($|\mu| < 2t$, $\Delta \neq 0$): Inter-site Majorana pairing leaves $\hat{\gamma}_1$ and $\hat{\gamma}_{2N}$ unpaired — **Majorana zero modes** with exactly zero energy.

---

In [ ]:
# Compare spectrum in trivial and topological phases
N = 12
n_levels = 6

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

params = [
    (3.0, 1.0, 1.0, 'Trivial ($\\mu=3t$)', axes[0]),
    (0.0, 1.0, 1.0, 'Topological ($\\mu=0$)', axes[1]),
]

for mu, t, Delta, title, ax in params:
    H = build_kitaev(N, mu=mu, t=t, Delta=Delta)
    evals = np.sort(np.linalg.eigvalsh(H.toarray()))
    mid = len(evals) // 2
    show = evals[mid - n_levels // 2: mid + n_levels // 2]
    ax.plot(range(len(show)), show, 'o', markersize=10)
    ax.axhline(0, color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel('Level index')
    ax.set_ylabel('Energy')
    ax.set_title(title)
    gap = abs(show[len(show)//2] - show[len(show)//2 - 1])
    ax.annotate(f'gap = {gap:.2e}', xy=(0.05, 0.95), xycoords='axes fraction',
                fontsize=10, verticalalignment='top')

plt.suptitle(f'Energy spectrum near zero, N={N}', y=1.02)
plt.tight_layout()
plt.show()

## 4. The Topological Phase Transition

The transition at $|\mu| = 2t$ is marked by the bulk gap closing. There is no local order parameter — both phases have the same spatial symmetry. The distinction is encoded in a **topological invariant**.

---

In [ ]:
# Gap as a function of mu/t: closes at |mu| = 2t
N = 14
mus = np.linspace(-4, 4, 80)
gaps = []

for mu in mus:
    H = build_kitaev(N, mu=mu, t=1.0, Delta=1.0)
    evals = np.sort(np.linalg.eigvalsh(H.toarray()))
    mid = len(evals) // 2
    gaps.append(abs(evals[mid] - evals[mid - 1]))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(mus, gaps, 'b-')
ax.axvline(-2, color='red', linestyle='--', alpha=0.7, label=r'$\mu = -2t$')
ax.axvline(+2, color='red', linestyle='--', alpha=0.7, label=r'$\mu = +2t$')
ax.fill_betweenx([0, max(gaps)], -2, 2, alpha=0.08, color='green', label='Topological')
ax.set_xlabel(r'$\mu / t$')
ax.set_ylabel('Bulk gap')
ax.set_title(f'Gap closing at topological phase transition, N={N}')
ax.legend()
plt.tight_layout()
plt.show()

## 5. The Topological Invariant

The $\mathbb{Z}_2$ topological invariant of the Kitaev chain can be computed from the momentum-space Hamiltonian. Define $h(k) = -(\mu + 2t\cos k)\tau_z + 2\Delta\sin k\,\tau_y$ where $\tau$ acts in particle-hole space. The **winding number** of $h(k)$ as $k$ traverses the Brillouin zone is 0 in the trivial phase and 1 in the topological phase.

---

In [ ]:
def winding_number(mu, t, Delta, n_k=1000):
    """Compute winding number of the Kitaev chain momentum-space Hamiltonian."""
    ks = np.linspace(0, 2 * np.pi, n_k, endpoint=False)
    # d-vector: h(k) = d_z * tau_z + d_y * tau_y
    d_z = -(mu + 2 * t * np.cos(ks))
    d_y = 2 * Delta * np.sin(ks)
    # Winding = (1/2pi) * integral of d(theta)/dk dk
    theta = np.arctan2(d_y, d_z)
    dtheta = np.diff(np.unwrap(theta))
    return round(np.sum(dtheta) / (2 * np.pi))


print("Topological invariant (winding number) vs μ/t:")
print(f"{'μ/t':>8}  {'winding':>8}  {'phase':>12}")
print("-" * 35)
for mu_over_t in [-3.0, -2.5, -1.5, 0.0, 1.5, 2.5, 3.0]:
    w = winding_number(mu=mu_over_t, t=1.0, Delta=1.0)
    phase = 'Topological' if w == 1 else 'Trivial'
    print(f"{mu_over_t:>8.1f}  {w:>8}  {phase:>12}")

## 6. Bulk-Boundary Correspondence

The topological invariant of the **bulk** determines the number of zero-energy modes at the **boundary**:
- Winding number $= 0$: no boundary modes.
- Winding number $= 1$: one Majorana zero mode at each end.

Protection: gapping a Majorana zero mode requires a term coupling $\hat{\gamma}_1$ and $\hat{\gamma}_{2N}$ — exponentially suppressed in chain length.

---

In [ ]:
# Visualise the Majorana zero mode wavefunction
N = 20
H_topo = build_kitaev(N, mu=0.0, t=1.0, Delta=1.0)
H_triv = build_kitaev(N, mu=3.0, t=1.0, Delta=1.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, H, title in [(axes[0], H_topo, 'Topological ($\\mu=0$)'),
                      (axes[1], H_triv, 'Trivial ($\\mu=3t$)')]:
    evals, evecs = eigsh(H, k=4, which='SM')  # smallest magnitude eigenvalues
    idx = np.argsort(np.abs(evals))
    # Zero mode: the eigenstate with energy closest to 0
    zero_mode = evecs[:, idx[0]]
    # Site-resolved probability: sum over all states with occupation at site i
    prob = np.zeros(N)
    for state in range(2**N):
        weight = zero_mode[state]**2
        for i in range(N):
            if (state >> i) & 1:
                prob[i] += weight

    ax.bar(range(N), prob, color='tab:blue' if 'Topological' in title else 'tab:orange')
    ax.set_xlabel('Site $i$')
    ax.set_ylabel('Local weight')
    ax.set_title(f'{title}\n$E_0 = {evals[idx[0]]:.2e}$')

plt.suptitle('Zero-mode wavefunction: localised at ends (topological) vs delocalised (trivial)', y=1.02)
plt.tight_layout()
plt.show()

## 7. The Non-Local Qubit

The two Majorana zero modes combine into a single complex fermion $\hat{f} = \frac{1}{2}(\hat{\gamma}_1 + i\,\hat{\gamma}_{2N})$, which is either occupied or empty — a **qubit**. This qubit is non-local: its two states differ in occupation of a mode split between the two ends of the chain. Any local perturbation affects only one Majorana operator and cannot rotate the qubit state. This is the basis for **topological quantum computing**.

---

## 8. Computing the Phase Diagram

We map out the phase diagram in the $(\mu/t, \Delta/t)$ plane using the bulk gap and winding number.

---

In [ ]:
# Phase diagram: winding number in (mu/t, Delta/t) plane
mu_vals = np.linspace(-3.5, 3.5, 50)
delta_vals = np.linspace(0.05, 2.5, 40)

winding = np.zeros((len(delta_vals), len(mu_vals)))

for j, mu in enumerate(mu_vals):
    for i, delta in enumerate(delta_vals):
        winding[i, j] = winding_number(mu=mu, t=1.0, Delta=delta)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.pcolormesh(mu_vals, delta_vals, winding, cmap='RdYlGn', vmin=-0.5, vmax=1.5)
plt.colorbar(im, ax=ax, label='Winding number')
ax.set_xlabel(r'$\mu / t$')
ax.set_ylabel(r'$\Delta / t$')
ax.set_title('Kitaev chain phase diagram')
ax.axvline(-2, color='white', linestyle='--', linewidth=1.5, alpha=0.8)
ax.axvline(+2, color='white', linestyle='--', linewidth=1.5, alpha=0.8)
ax.text(-1.0, 1.5, 'Topological\n(W=1)', ha='center', color='white', fontsize=11, fontweight='bold')
ax.text(-3.0, 1.5, 'Trivial\n(W=0)', ha='center', color='black', fontsize=11)
ax.text(+3.0, 1.5, 'Trivial\n(W=0)', ha='center', color='black', fontsize=11)
plt.tight_layout()
plt.show()

## 9. Where the Series Has Brought You

| Lecture | Physics | Method |
|---|---|---|
| 01 | 2D Ising model, thermal phase transition | Metropolis Monte Carlo |
| 02 | Critical exponents, universality, FSS | Finite-size scaling |
| 03 | Quantum spins, TFIM, quantum-to-classical mapping | Pauli matrices, tensor products |
| 04 | Quantum phase transition, spectral gap, scaling collapse | ED + FSS |
| 05 | Many-body Hilbert space, sparse Hamiltonian, Lanczos | Exact diagonalisation I |
| 06 | Energy spectrum, correlations, susceptibility, phase diagram | Exact diagonalisation II |
| 07 | Bipartite entanglement, area law, central charge | Schmidt decomposition, SVD |
| 08 | MPS representation, SVD truncation, canonical form | Tensor networks I |
| 09 | DMRG algorithm, variational optimisation, convergence | Tensor networks II |
| 10 | TeNPy, large-scale DMRG, phase diagram at $L\sim500$ | Tensor networks III |
| 11 | Topological order, Majorana fermions, bulk-boundary correspondence | ED + TeNPy |

The series has taken you from thermal fluctuations in a classical model all the way to topological quantum phases and Majorana fermions — a journey that mirrors the development of condensed matter physics over the past three decades. The tools you have built — Monte Carlo, finite-size scaling, exact diagonalisation, tensor networks — are the same ones used in active research today.

---